In [0]:
from datetime import datetime
from pyspark.sql.functions import month, year
def bronze_ingestion(schema,landing_location,checkpoint_location,table_name,archive_location):
    try:
        exec_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        archive_directory=f'{archive_location}/{exec_timestamp}'
        dbutils.fs.mkdirs(f'{archive_location}/{exec_timestamp}')

        

        bronze_df=spark.readStream.format("json") \
            .schema(schema)\
                .option("cleanSource","archive").option("sourceArchiveDir",f"{archive_directory}")\
                .option("maxFilesPerTrigger", 5)\
                .load(f"{landing_location}")
        
        bronze_df = bronze_df.withColumn("month", month("event_time")).withColumn("year", year("event_time"))


        streaming_query_bronze=bronze_df.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation",f"{checkpoint_location}")\
            .partitionBy("year", "month") \
            .trigger(availableNow=True)\
            .toTable(f"cosmetic_history.bronze.{table_name}")
        return streaming_query_bronze
    except Exception as e:
        raise Exception(f"Error: {e}")

if __name__=="__main__":
    schema="""
    event_time timestamp,
    event_type string,
    user_id int,
    user_session string,
    product_info struct<product_id: string, category_id: string, category_code: string, brand: string, price: double>
    """
    landing_location='/Volumes/cosmetic_history/landing/landing_zone/source_files'
    checkpoint_location='/Volumes/cosmetic_history/checkpoint/cosmetic_history_checkpoints/bronze_checkpoint'
    table_name='Users_Products_Info_Bronze'
    archive_location='/Volumes/cosmetic_history/landing/landing_zone/Archive/'
    streaming_query=bronze_ingestion(schema,landing_location,checkpoint_location,table_name,archive_location)
    streaming_query.awaitTermination()
    print("Done")

Done
